In [ ]:
# ============================================================
# CELL 1: GPU Check & Environment Setup
# ============================================================
import torch
import os

print("=" * 50)
print("GPU AVAILABILITY CHECK")
print("=" * 50)
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU Available: {gpu_name}")
    print(f"   Total VRAM   : {total_mem:.2f} GB")
    DEVICE = torch.device("cuda")
else:
    print("⚠️  No GPU found — using CPU (training will be slow)")
    DEVICE = torch.device("cpu")

print(f"\nUsing device: {DEVICE}")

In [ ]:
# ============================================================
# CELL 2: Imports
# ============================================================
import os, random, shutil, warnings, time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from pathlib import Path
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

from sklearn.metrics import classification_report, confusion_matrix

warnings.filterwarnings("ignore")

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

print("✅ All imports successful.")

In [ ]:
# ============================================================
# CELL 3: Configuration
# ============================================================

# ── Paths ──────────────────────────────────────────────────
DATASET_PATH   = Path("/kaggle/input/datasets/bapdesilva/betel-leaf-quality-dataset-2025/betel-quality-ds")
AUGMENTED_PATH = Path("/kaggle/working/augmented_dataset")
FINAL_PATH     = Path("/kaggle/working/final_dataset")

# ── Dataset ────────────────────────────────────────────────
TARGET_PER_CLASS = 2500
VAL_SPLIT        = 0.20
TEST_SPLIT       = 0.05 
IMG_SIZE         = 224

# ── Training hyper-parameters ──────────────────────────────
BATCH_SIZE   = 32
NUM_EPOCHS   = 30
LR_HEAD      = 1e-3
LR_FINE      = 1e-4
WEIGHT_DECAY = 1e-4
PATIENCE     = 7
UNFREEZE_PCT = 0.30

CLASS_NAMES = [
    "Grade A Quality",
    "Grade B Quality",
    "Grade C Quality",
    "Grade D Quality",
    "Grade E Quality",
]
NUM_CLASSES = len(CLASS_NAMES)

print("✅ Configuration set.")
print(f"   Dataset path  : {DATASET_PATH}")
print(f"   Target/class  : {TARGET_PER_CLASS}")
print(f"   Val split     : {VAL_SPLIT*100:.0f}%")
print(f"   test split     : {TEST_SPLIT*100:.0f}%")
print(f"   Num classes   : {NUM_CLASSES}")
print(f"   Device        : {DEVICE}")

In [ ]:
# ============================================================
# CELL 4: Inspect Dataset Structure
# ============================================================
print("Dataset folder structure:")
print("=" * 50)

found_classes = []
for item in sorted(DATASET_PATH.iterdir()):
    if item.is_dir():
        exts  = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".webp"}
        count = len([f for f in item.iterdir()
                     if f.suffix.lower() in exts])
        print(f"  [{item.name}]  →  {count} images")
        found_classes.append(item.name)

print(f"\nFound {len(found_classes)} class folders:")
for c in found_classes:
    print(f"   {c}")

# Auto-update CLASS_NAMES if folder names differ slightly
if set(found_classes) != set(CLASS_NAMES):
    print("\n⚠️  Folder names differ from CLASS_NAMES config.")
    print("   Updating CLASS_NAMES to match actual folders...")
    CLASS_NAMES = sorted(found_classes)
    NUM_CLASSES = len(CLASS_NAMES)
    print(f"   Updated CLASS_NAMES: {CLASS_NAMES}")

In [ ]:
# ============================================================
# CELL 5: Offline Augmentation (run ONCE — before training)
# ============================================================

aug_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(degrees=30),
    transforms.ColorJitter(brightness=0.3, contrast=0.3,
                           saturation=0.2, hue=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1),
                            scale=(0.85, 1.15), shear=10),
    transforms.RandomPerspective(distortion_scale=0.3, p=0.4),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.5)),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
])

def augment_class_folder(src_folder: Path, dst_folder: Path, target: int):
    dst_folder.mkdir(parents=True, exist_ok=True)
    exts     = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".webp"}
    originals = [p for p in src_folder.iterdir()
                 if p.suffix.lower() in exts]

    if not originals:
        print(f"  ⚠️  No images found in {src_folder} — skipping.")
        return 0

    count = 0

    # 1. Copy originals
    for p in originals:
        shutil.copy(p, dst_folder / p.name)
        count += 1

    # 2. Augment until target reached
    idx = 0
    while count < target:
        src_img = originals[idx % len(originals)]
        img     = Image.open(src_img).convert("RGB")
        aug_img = aug_transform(img)
        aug_img.save(dst_folder / f"aug_{count:05d}.jpg")
        count += 1
        idx   += 1

    return count


print("=" * 55)
print(f"OFFLINE AUGMENTATION  (target = {TARGET_PER_CLASS} images/class)")
print("=" * 55)

total_generated = 0
for cls in CLASS_NAMES:
    src = DATASET_PATH / cls
    dst = AUGMENTED_PATH / cls

    if not src.exists():
        print(f"  ❌ Folder not found: {src}")
        continue

    n_orig = len([f for f in src.iterdir()
                  if f.suffix.lower() in
                  {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".webp"}])
    print(f"\n  [{cls}]  originals: {n_orig}", end=" → ")

    n_final = augment_class_folder(src, dst, TARGET_PER_CLASS)
    print(f"total after augmentation: {n_final}")
    total_generated += n_final

print(f"\n✅ Augmentation complete.  Grand total: {total_generated} images")